In [ ]:
import pandas as pd

Take the reporting company from the ghg file and then append it to ab emissions

In [ ]:
ab_emissions = pd.read_csv("../../lat_long_to_lsd/ab_emissions_with_lsd.csv")
ghg = pd.read_excel("../../data_sets/PDGES-GHGRP-GHGEmissionsGES-2004-Present.xlsx")
ghrp_with_company = ghg.iloc[:, [0, 13, 15, 16]].drop_duplicates().reset_index(drop=True)

In [ ]:
ghrp_with_company["GHGRP ID No. / No d'identification du PDGES"]

In [ ]:
ab_emissions_with_company = ab_emissions.merge(ghrp_with_company, on="GHGRP ID No. / No d'identification du PDGES", how="left")

In [ ]:
ab_emissions_with_company.to_csv("ab_emissions_with_company.csv")

Okay now we will clean the emissions data

In [ ]:
ab_emissions_with_company.columns = [
    col.split('/')[0].strip().lower().replace(' ', '_').replace('(', '_').replace(')', '') for col in ab_emissions_with_company.columns
]

# change id no to just id
ab_emissions_with_company.rename(columns={'ghgrp_id_no.': 'ghgrp_id'}, inplace=True)

In [ ]:
ab_emissions_with_company.info()

In [ ]:
# Simple comparison
ghg_facilities = ab_emissions_with_company[['ghgrp_id', 'reference_year', 'facility_name', 'dls', 'reporting_company_legal_name', 'total_emissions__tonnes_co2e']]\
    .drop_duplicates().reset_index(drop=True)

In [ ]:
facilities_data = pd.read_csv("../data_cleaned/facilities/facilities.csv")

In [ ]:
print("Number of facilities from original data:", facilities_data.index.size)
print("Number of ghg facilities", ghg_facilities.index.size)

In [ ]:
facilities_data

In [ ]:
merged = facilities_data.merge(ghg_facilities, left_on="location", right_on="dls", how="inner")

In [ ]:
merged.info()

In [ ]:
merged.groupby(['id', 'reference_year'])[['total_emissions__tonnes_co2e']].sum().reset_index()

In [ ]:
import os
path = "../data_cleaned/yearly_total_emissions_co2"
if not os.path.exists(path):
    os.mkdir(path)
merged.to_csv(path + "/yearly_total_emissions_co2.csv")